# OOS — AutoML argmax (недостающие прогоны)

Код из репозитория **вшит в ноутбук** (без git). Стек: `prepare_data` → `experiment_runner` → AutoGluon / H2O / LAMA.

**Сейчас в очереди: 12 jobs** (h2o: 3, lama: 9)

- AutoGluon: 12/12 уже в `metrics.json` — **не запускается**
- Пропуск успешных прогонов: встроенный список + файл `automl_frameworks_argmax_metrics.json` на Kaggle
- Логи: только `RUN_START` / `RUN_FINISH`
- Укажите **HF_TOKEN** в ячейке 2 (или Kaggle Secrets)
- GPU **T4**, Internet On


In [ ]:
# 1. Setup
import os, sys, subprocess, json, logging, warnings
from pathlib import Path

for _k, _v in {
    "OMP_NUM_THREADS": "1", "OPENBLAS_NUM_THREADS": "1", "MKL_NUM_THREADS": "1",
    "NUMEXPR_NUM_THREADS": "1", "TOKENIZERS_PARALLELISM": "false",
    "HF_HUB_DISABLE_PROGRESS_BARS": "1", "TRANSFORMERS_VERBOSITY": "error",
    "OOS_METRICS_LOG": "compact", "OOS_QUIET_FIT": "1", "OOS_H2O_QUIET": "1",
    "H2O_MAX_MEM": "6G", "H2O_NTHREADS": "2",
}.items():
    os.environ.setdefault(_k, _v)

warnings.filterwarnings("ignore")
logging.disable(logging.CRITICAL)
os.environ.setdefault("PYTHONWARNINGS", "ignore")

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "datasets>=2.14", "sentence-transformers>=2.7", "scikit-learn>=1.3",
    "pandas", "h2o>=3.44", "autogluon.tabular>=1.1", "lightautoml>=0.3.8"],
    check=False)

WORKDIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd() / "kaggle_working"
WORKDIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# 2. HuggingFace token + paths
HF_TOKEN = ""  # <-- вставьте токен или оставьте пустым для Kaggle Secrets

if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        for _name in ("HF_TOKEN", "HUGGINGFACE_HUB_TOKEN"):
            try:
                HF_TOKEN = UserSecretsClient().get_secret(_name)
                if HF_TOKEN:
                    break
            except Exception:
                pass
    except Exception:
        pass

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN
    try:
        from huggingface_hub import login
        login(token=HF_TOKEN, add_to_git_credential=False)
    except Exception:
        pass

BUNDLE_ROOT = WORKDIR / "oos_bundle"
METRICS_JSON = WORKDIR / "automl_frameworks_argmax_metrics.json"
SOURCE = "deeppavlov"
EMBEDDER = "intfloat/multilingual-e5-large-instruct"


In [ ]:
# 3. Install bundled repo code (same files as tasks/oos_detection/)
import json
BUNDLE_FILES = json.loads('{"tasks/__init__.py": "", "tasks/oos_detection/__init__.py": "", "tasks/oos_detection/src/__init__.py": "", "tasks/oos_detection/src/metrics.py": "\\"\\"\\"\\nМетрики для оценки OOS-детекции.\\n\\nОсновные (прямое сравнение с литературой):\\n  - oos_recall     : TPR на OOS-классе (ADB, DETER, AutoIntent)\\n  - in_domain_acc  : accuracy на in-scope (AutoIntent Table 3: 96.13)\\n  - f1_oos         : F1 на OOS-классе (AutoIntent Table 3: 76.79)\\n\\nВспомогательные:\\n  - auroc          : Area Under ROC Curve (EMNLP Industry 2024)\\n  - au_ioc         : Area Under IOC curve (Springer Applied Intelligence 2024)\\n  - latency_ms     : время инференса (для guardrail-аргументации)\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\nimport os\\nimport time\\n\\nimport numpy as np\\nfrom sklearn.metrics import (\\n    auc,\\n    roc_auc_score,\\n    f1_score,\\n    recall_score,\\n    accuracy_score,\\n)\\n\\n\\ndef oos_recall(\\n    y_true: np.ndarray,\\n    y_pred: np.ndarray,\\n    oos_label: int = -1,\\n) -> float:\\n    \\"\\"\\"Recall на OOS-классе (TPR).\\"\\"\\"\\n    y_true_bin = (np.asarray(y_true) == oos_label).astype(int)\\n    y_pred_bin = (np.asarray(y_pred) == oos_label).astype(int)\\n    return float(recall_score(y_true_bin, y_pred_bin, zero_division=0))\\n\\n\\ndef in_domain_accuracy(\\n    y_true: np.ndarray,\\n    y_pred: np.ndarray,\\n    oos_label: int = -1,\\n) -> float:\\n    \\"\\"\\"Accuracy только на in-scope примерах.\\"\\"\\"\\n    y_true = np.asarray(y_true)\\n    y_pred = np.asarray(y_pred)\\n    mask = y_true != oos_label\\n    if mask.sum() == 0:\\n        return 0.0\\n    return float(accuracy_score(y_true[mask], y_pred[mask]))\\n\\n\\ndef f1_oos(\\n    y_true: np.ndarray,\\n    y_pred: np.ndarray,\\n    oos_label: int = -1,\\n) -> float:\\n    \\"\\"\\"F1-score на OOS-классе.\\"\\"\\"\\n    y_true_bin = (np.asarray(y_true) == oos_label).astype(int)\\n    y_pred_bin = (np.asarray(y_pred) == oos_label).astype(int)\\n    return float(f1_score(y_true_bin, y_pred_bin, zero_division=0))\\n\\n\\ndef auroc(\\n    y_true: np.ndarray,\\n    y_scores: np.ndarray,\\n    oos_label: int = -1,\\n) -> float:\\n    \\"\\"\\"Area Under ROC Curve для бинарной задачи OOS vs in-scope.\\"\\"\\"\\n    y_true_binary = (np.asarray(y_true) == oos_label).astype(int)\\n    if len(np.unique(y_true_binary)) < 2:\\n        return 0.5\\n    return float(roc_auc_score(y_true_binary, y_scores))\\n\\n\\ndef au_ioc(\\n    y_true: np.ndarray,\\n    y_scores: np.ndarray,\\n    y_pred_intent: np.ndarray | None = None,\\n    oos_label: int = -1,\\n    n_thresholds: int = 100,\\n) -> float:\\n    \\"\\"\\"\\n    Area Under In-scope/Out-of-scope Characteristic curve.\\n    X-axis: in-domain accuracy, Y-axis: OOS recall.\\n    \\"\\"\\"\\n    y_true = np.asarray(y_true)\\n    y_scores = np.asarray(y_scores)\\n\\n    oos_mask = y_true == oos_label\\n    inscope_mask = ~oos_mask\\n    n_oos, n_inscope = oos_mask.sum(), inscope_mask.sum()\\n\\n    if n_oos == 0 or n_inscope == 0:\\n        return 0.5\\n\\n    # Vectorized: thresholds x samples\\n    thresholds = np.linspace(y_scores.min(), y_scores.max(), n_thresholds)\\n    pred_oos = y_scores[None, :] >= thresholds[:, None]  # (n_thresh, n_samples)\\n\\n    # OOS recall per threshold\\n    oos_recalls = (pred_oos & oos_mask).sum(axis=1) / n_oos\\n\\n    # In-domain accuracy per threshold\\n    inscope_not_oos = inscope_mask & ~pred_oos  # (n_thresh, n_samples)\\n    if y_pred_intent is not None:\\n        y_pred_intent = np.asarray(y_pred_intent)\\n        correct = inscope_not_oos & (y_pred_intent == y_true)\\n        in_domain_accs = correct.sum(axis=1) / n_inscope\\n    else:\\n        in_domain_accs = inscope_not_oos.sum(axis=1) / n_inscope\\n\\n    # Sort by x and compute AUC\\n    order = np.argsort(in_domain_accs)\\n    auc_value = float(auc(in_domain_accs[order], oos_recalls[order]))\\n    return max(0.0, min(1.0, auc_value))\\n\\n\\ndef get_oos_scores_from_pipeline(pipeline, texts: list[str]) -> np.ndarray:\\n    \\"\\"\\"\\n    Get continuous OOS scores from AutoIntent pipeline.\\n    Uses predict_with_metadata to get class probability scores,\\n    then computes OOS score as 1 - max(class_scores).\\n    Falls back to binary scores if continuous unavailable.\\n    Returns array of shape (n_samples,).\\n    \\"\\"\\"\\n    if hasattr(pipeline, \'predict_with_metadata\'):\\n        try:\\n            results = pipeline.predict_with_metadata(texts)\\n            if hasattr(results, \'utterances\') and results.utterances:\\n                all_scores = [utt.score for utt in results.utterances\\n                              if hasattr(utt, \'score\') and utt.score is not None]\\n                if all_scores:\\n                    return 1 - np.array(all_scores).max(axis=1)\\n        except Exception:\\n            pass\\n    # fallback\\n    preds = pipeline.predict(texts)\\n    return np.array([1.0 if p is None else 0.0 for p in preds])\\n\\n\\ndef measure_latency(\\n    model,\\n    texts: list[str],\\n    n_warmup: int = 5,\\n    n_runs: int = 50,\\n) -> float:\\n    \\"\\"\\"Среднее время инференса на 1 запрос в миллисекундах.\\"\\"\\"\\n    if os.environ.get(\\"OOS_METRICS_LOG\\", \\"\\").lower() == \\"compact\\":\\n        n_warmup, n_runs = 2, 5\\n    texts = texts[:100]\\n\\n    # Warmup\\n    for i in range(n_warmup):\\n        model.predict([texts[i % len(texts)]])\\n\\n    # Measure\\n    times = []\\n    for i in range(n_runs):\\n        text = texts[i % len(texts)]\\n        start = time.perf_counter()\\n        model.predict([text])\\n        times.append((time.perf_counter() - start) * 1000)\\n\\n    return float(np.mean(times))\\n\\n\\ndef compute_all_metrics(\\n    y_true: np.ndarray,\\n    y_scores: np.ndarray,\\n    y_pred: np.ndarray,\\n    oos_label: int = -1,\\n) -> dict:\\n    \\"\\"\\"Вычисляет все метрики и возвращает словарь.\\"\\"\\"\\n    return {\\n        \\"oos_recall\\": oos_recall(y_true, y_pred, oos_label),\\n        \\"in_domain_acc\\": in_domain_accuracy(y_true, y_pred, oos_label),\\n        \\"f1_oos\\": f1_oos(y_true, y_pred, oos_label),\\n        \\"auroc\\": auroc(y_true, y_scores, oos_label),\\n        \\"au_ioc\\": au_ioc(y_true, y_scores, y_pred, oos_label),\\n    }\\n", "tasks/oos_detection/src/evaluation.py": "\\"\\"\\"\\nЕдиный интерфейс оценки моделей.\\nВсе модели (бейзлайн, SOTA, AutoIntent) оцениваются через один класс,\\nчтобы результаты были сопоставимы.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\nfrom dataclasses import dataclass, field, asdict\\nfrom pathlib import Path\\nimport json\\nimport time\\nfrom typing import Protocol, runtime_checkable\\n\\nimport numpy as np\\n\\nfrom .metrics import compute_all_metrics, measure_latency\\n\\n\\n@runtime_checkable\\nclass OOSModel(Protocol):\\n    \\"\\"\\"Protocol for OOS detection models.\\"\\"\\"\\n\\n    def predict(self, texts: list[str]) -> np.ndarray:\\n        \\"\\"\\"Return predicted labels (-1 for OOS, 0..N for intents).\\"\\"\\"\\n        ...\\n\\n    def predict_proba(self, texts: list[str]) -> np.ndarray:\\n        \\"\\"\\"Return OOS probability scores (higher = more likely OOS).\\"\\"\\"\\n        ...\\n\\n\\n@dataclass\\nclass EvaluationResult:\\n    \\"\\"\\"\\n    Результат оценки одной модели на тест-сплите.\\n\\n    Метрики соответствуют стандарту OOS NLP-литературы.\\n    Основные три (oos_recall, in_domain_acc, f1_oos) позволяют\\n    напрямую сравниться с AutoIntent Table 3, ADB, DETER.\\n    \\"\\"\\"\\n    model_name: str\\n    mode: str                      # \\"full\\" | \\"10shot\\" | \\"20shot\\" | \\"50shot\\"\\n\\n    # Основные метрики — прямое сравнение с литературой\\n    oos_recall: float = 0.0        # TPR на OOS; стандарт в ADB, DETER, AutoIntent\\n    in_domain_acc: float = 0.0     # accuracy на in-scope; AutoIntent Table 3: 96.13\\n    f1_oos: float = 0.0            # F1 на OOS; AutoIntent Table 3: 76.79\\n\\n    # Вспомогательные метрики\\n    auroc: float = 0.0             # EMNLP Industry 2024\\n    au_ioc: float = 0.0            # Springer Applied Intelligence 2024\\n    latency_ms: float = 0.0        # для guardrail-аргументации\\n\\n    # Мета-информация\\n    n_shots: int | None = None     # None = full train\\n    seed: int | None = None\\n    is_reference: bool = False     # True для guardrail reference (Chua 2024)\\n    extra: dict = field(default_factory=dict)\\n\\n    def to_dict(self) -> dict:\\n        \\"\\"\\"Convert to dictionary for JSON serialization.\\"\\"\\"\\n        return asdict(self)\\n\\n\\nclass Evaluator:\\n    \\"\\"\\"\\n    Оценивает модель на тест-сплите и сохраняет результаты.\\n\\n    Использование:\\n        evaluator = Evaluator(test_data, results_dir)\\n        result = evaluator.evaluate(model, model_name=\\"autointent_10shot\\")\\n        evaluator.save(result)\\n        evaluator.print_report()\\n    \\"\\"\\"\\n\\n    def __init__(self, test_data: dict, results_dir: Path):\\n        \\"\\"\\"\\n        Args:\\n            test_data: {\\"texts\\": list[str], \\"labels\\": list[int]}\\n                       где -1 = OOS, 0..N = in-scope intents\\n            results_dir: директория для сохранения результатов\\n        \\"\\"\\"\\n        self.texts = test_data[\\"texts\\"]\\n        self.labels = np.array(test_data[\\"labels\\"])\\n        self.results_dir = Path(results_dir)\\n        self.results_dir.mkdir(parents=True, exist_ok=True)\\n        self.metrics_file = self.results_dir / \\"metrics.json\\"\\n        self.oos_label = -1\\n\\n    def evaluate(\\n        self,\\n        model: OOSModel,\\n        model_name: str,\\n        mode: str = \\"full\\",\\n        n_shots: int | None = None,\\n        seed: int | None = None,\\n        is_reference: bool = False,\\n        measure_latency_flag: bool = True,\\n        extra: dict | None = None,\\n    ) -> EvaluationResult:\\n        \\"\\"\\"\\n        Прогоняет модель на тест-сплите, вычисляет все метрики,\\n        измеряет latency.\\n\\n        Args:\\n            model: объект с методами predict() и predict_proba_oos()\\n            model_name: название модели для отчёта\\n            mode: \\"full\\" | \\"10shot\\" | \\"20shot\\" | \\"50shot\\"\\n            n_shots: число примеров на класс (для few-shot)\\n            seed: random seed (для few-shot)\\n            is_reference: True для guardrail reference model\\n            measure_latency_flag: измерять ли latency\\n\\n        Returns:\\n            EvaluationResult с заполненными метриками\\n        \\"\\"\\"\\n        # Get predictions\\n        y_pred = model.predict(self.texts)\\n        y_scores = model.predict_proba(self.texts)\\n\\n        # Compute metrics\\n        metrics = compute_all_metrics(\\n            y_true=self.labels,\\n            y_scores=y_scores,\\n            y_pred=y_pred,\\n            oos_label=self.oos_label,\\n        )\\n\\n        # Measure latency (first 100 texts for speed)\\n        latency = 0.0\\n        if measure_latency_flag:\\n            latency = measure_latency(model, self.texts[:100])\\n\\n        return EvaluationResult(\\n            model_name=model_name,\\n            mode=mode,\\n            oos_recall=metrics[\\"oos_recall\\"],\\n            in_domain_acc=metrics[\\"in_domain_acc\\"],\\n            f1_oos=metrics[\\"f1_oos\\"],\\n            auroc=metrics[\\"auroc\\"],\\n            au_ioc=metrics[\\"au_ioc\\"],\\n            latency_ms=latency,\\n            n_shots=n_shots,\\n            seed=seed,\\n            is_reference=is_reference,\\n            extra=extra or {},\\n        )\\n\\n    def save(self, result: EvaluationResult) -> None:\\n        \\"\\"\\"\\n        Сохраняет результат в results/metrics.json.\\n\\n        Если результат с таким же (model_name, mode, seed) уже существует,\\n        он будет обновлён. Иначе — добавлен.\\n        \\"\\"\\"\\n        # Load existing results\\n        results = []\\n        if self.metrics_file.exists():\\n            with open(self.metrics_file, \\"r\\", encoding=\\"utf-8\\") as f:\\n                results = json.load(f)\\n\\n        # Create key for deduplication (include source to separate datasets)\\n        result_dict = result.to_dict()\\n        source = result.extra.get(\\"source\\") if result.extra else None\\n        key = (result.model_name, result.mode, result.seed, source)\\n\\n        # Find and update existing, or append new\\n        found = False\\n        for i, r in enumerate(results):\\n            existing_source = r.get(\\"extra\\", {}).get(\\"source\\")\\n            existing_key = (r[\\"model_name\\"], r[\\"mode\\"], r.get(\\"seed\\"), existing_source)\\n            if existing_key == key:\\n                results[i] = result_dict\\n                found = True\\n                break\\n\\n        if not found:\\n            results.append(result_dict)\\n\\n        # Save\\n        with open(self.metrics_file, \\"w\\", encoding=\\"utf-8\\") as f:\\n            json.dump(results, f, ensure_ascii=False, indent=2)\\n\\n    def load_results(self) -> list[dict]:\\n        \\"\\"\\"Загружает все сохранённые результаты.\\"\\"\\"\\n        if not self.metrics_file.exists():\\n            return []\\n        with open(self.metrics_file, \\"r\\", encoding=\\"utf-8\\") as f:\\n            return json.load(f)\\n\\n    def print_report(self) -> None:\\n        \\"\\"\\"Печатает сравнительную таблицу всех сохранённых результатов.\\"\\"\\"\\n        results = self.load_results()\\n        if not results:\\n            print(\\"No results saved yet.\\")\\n            return\\n\\n        # Header\\n        print(\\"\\\\n\\" + \\"=\\" * 115)\\n        print(\\"OOS Detection Evaluation Report\\")\\n        print(\\"=\\" * 115)\\n        print(\\n            f\\"{\'Model\':<25} {\'Source\':<10} {\'Mode\':<10} \\"\\n            f\\"{\'OOS Rec\':>8} {\'InD Acc\':>8} {\'F1 OOS\':>8} \\"\\n            f\\"{\'AUROC\':>8} {\'AU-IOC\':>8} {\'Lat(ms)\':>8}\\"\\n        )\\n        print(\\"-\\" * 115)\\n\\n        # Sort by f1_oos descending\\n        sorted_results = sorted(results, key=lambda x: -x[\\"f1_oos\\"])\\n\\n        for r in sorted_results:\\n            # Mark reference models\\n            name = r[\\"model_name\\"]\\n            if r.get(\\"is_reference\\"):\\n                name += \\" [ref]\\"\\n\\n            # Get source from extra\\n            source = r.get(\\"extra\\", {}).get(\\"source\\", \\"—\\")\\n\\n            print(\\n                f\\"{name:<25} {source:<10} {r[\'mode\']:<10} \\"\\n                f\\"{r[\'oos_recall\']:>8.4f} {r[\'in_domain_acc\']:>8.4f} {r[\'f1_oos\']:>8.4f} \\"\\n                f\\"{r[\'auroc\']:>8.4f} {r[\'au_ioc\']:>8.4f} {r[\'latency_ms\']:>8.2f}\\"\\n            )\\n\\n        print(\\"-\\" * 115)\\n        print(\\n            \\"Primary metrics: OOS Recall, In-Domain Acc, F1 OOS | \\"\\n            \\"Secondary: AUROC, AU-IOC, Latency\\"\\n        )\\n        print(\\"=\\" * 115 + \\"\\\\n\\")\\n", "tasks/oos_detection/src/data_utils.py": "\\"\\"\\"\\nУтилиты для загрузки подготовленных данных CLINC150.\\n\\nДанные должны быть предварительно созданы через:\\n    python scripts/prepare_data.py --source standard\\n    python scripts/prepare_data.py --source deeppavlov\\n\\nИсточники:\\n    - standard: github.com/clinc/oos-eval (100 OOS train)\\n    - deeppavlov: HuggingFace DeepPavlov/clinc150 (200 OOS train)\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport json\\nfrom pathlib import Path\\n\\n\\n# === Constants ===\\n\\nOOS_LABEL = -1\\nVALID_SOURCES = (\\"standard\\", \\"deeppavlov\\")\\nVALID_SPLITS = (\\"train\\", \\"validation\\", \\"test\\")\\n\\n# Default path to processed data\\n_DEFAULT_PROCESSED_DIR = Path(__file__).parent.parent / \\"data\\" / \\"processed\\"\\n\\n\\ndef _get_processed_dir(source: str, processed_dir: Path | None = None) -> Path:\\n    \\"\\"\\"Get path to processed data directory for a source.\\"\\"\\"\\n    if source not in VALID_SOURCES:\\n        raise ValueError(f\\"source must be one of {VALID_SOURCES}, got \'{source}\'\\")\\n\\n    base_dir = processed_dir or _DEFAULT_PROCESSED_DIR\\n    source_dir = base_dir / source\\n\\n    if not source_dir.exists():\\n        raise FileNotFoundError(\\n            f\\"Processed data not found: {source_dir}\\\\n\\"\\n            f\\"Run: python scripts/prepare_data.py --source {source}\\"\\n        )\\n\\n    return source_dir\\n\\n\\n# === Loading functions ===\\n\\ndef load_split(\\n    source: str,\\n    split: str,\\n    processed_dir: Path | None = None,\\n) -> dict:\\n    \\"\\"\\"\\n    Загружает сплит данных.\\n\\n    Args:\\n        source: \\"standard\\" | \\"deeppavlov\\"\\n        split: \\"train\\" | \\"validation\\" | \\"test\\"\\n        processed_dir: путь к data/processed/ (опционально)\\n\\n    Returns:\\n        {\\"texts\\": list[str], \\"labels\\": list[int]}\\n        OOS имеет label = -1\\n    \\"\\"\\"\\n    if split not in VALID_SPLITS:\\n        raise ValueError(f\\"split must be one of {VALID_SPLITS}, got \'{split}\'\\")\\n\\n    source_dir = _get_processed_dir(source, processed_dir)\\n    full_path = source_dir / \\"full.json\\"\\n\\n    with open(full_path, \\"r\\", encoding=\\"utf-8\\") as f:\\n        data = json.load(f)\\n\\n    return data[split]\\n\\n\\ndef load_fewshot(\\n    source: str,\\n    n_shots: int,\\n    seed: int,\\n    processed_dir: Path | None = None,\\n) -> dict:\\n    \\"\\"\\"\\n    Загружает few-shot train выборку.\\n\\n    Args:\\n        source: \\"standard\\" | \\"deeppavlov\\"\\n        n_shots: 10 | 20 | 50\\n        seed: 42 | 123 | 456\\n        processed_dir: путь к data/processed/ (опционально)\\n\\n    Returns:\\n        {\\"texts\\": list[str], \\"labels\\": list[int]}\\n    \\"\\"\\"\\n    source_dir = _get_processed_dir(source, processed_dir)\\n    fewshot_path = source_dir / \\"fewshot.json\\"\\n\\n    with open(fewshot_path, \\"r\\", encoding=\\"utf-8\\") as f:\\n        data = json.load(f)\\n\\n    n_key = f\\"n{n_shots}\\"\\n    seed_key = f\\"seed{seed}\\"\\n\\n    if n_key not in data:\\n        raise ValueError(f\\"n_shots={n_shots} not found. Available: {list(data.keys())}\\")\\n    if seed_key not in data[n_key]:\\n        raise ValueError(f\\"seed={seed} not found. Available: {list(data[n_key].keys())}\\")\\n\\n    return data[n_key][seed_key]\\n\\n\\ndef get_intents(\\n    source: str,\\n    processed_dir: Path | None = None,\\n) -> list[dict]:\\n    \\"\\"\\"\\n    Возвращает список интентов.\\n\\n    Args:\\n        source: \\"standard\\" | \\"deeppavlov\\"\\n        processed_dir: путь к data/processed/ (опционально)\\n\\n    Returns:\\n        [{\\"id\\": int, \\"name\\": str}, ...] отсортировано по id\\n    \\"\\"\\"\\n    source_dir = _get_processed_dir(source, processed_dir)\\n    meta_path = source_dir / \\"meta.json\\"\\n\\n    with open(meta_path, \\"r\\", encoding=\\"utf-8\\") as f:\\n        meta = json.load(f)\\n\\n    return meta[\\"intents\\"]\\n\\n\\ndef get_intent_names(\\n    source: str,\\n    processed_dir: Path | None = None,\\n) -> dict[int, str]:\\n    \\"\\"\\"\\n    Возвращает mapping label_id -> intent_name.\\n\\n    Args:\\n        source: \\"standard\\" | \\"deeppavlov\\"\\n        processed_dir: путь к data/processed/ (опционально)\\n\\n    Returns:\\n        {label_id: intent_name}\\n        Не включает OOS (label -1)\\n    \\"\\"\\"\\n    intents = get_intents(source, processed_dir)\\n    return {intent[\\"id\\"]: intent[\\"name\\"] for intent in intents}\\n\\n\\ndef load_meta(\\n    source: str,\\n    processed_dir: Path | None = None,\\n) -> dict:\\n    \\"\\"\\"\\n    Загружает метаданные датасета.\\n\\n    Args:\\n        source: \\"standard\\" | \\"deeppavlov\\"\\n        processed_dir: путь к data/processed/ (опционально)\\n\\n    Returns:\\n        dict с метаданными\\n    \\"\\"\\"\\n    source_dir = _get_processed_dir(source, processed_dir)\\n    meta_path = source_dir / \\"meta.json\\"\\n\\n    with open(meta_path, \\"r\\", encoding=\\"utf-8\\") as f:\\n        return json.load(f)\\n\\n\\ndef get_split_stats(\\n    source: str,\\n    processed_dir: Path | None = None,\\n) -> dict:\\n    \\"\\"\\"\\n    Возвращает статистику по сплитам.\\n\\n    Args:\\n        source: \\"standard\\" | \\"deeppavlov\\"\\n        processed_dir: путь к data/processed/ (опционально)\\n\\n    Returns:\\n        {split_name: {\\"total\\": int, \\"n_inscope\\": int, \\"n_oos\\": int}}\\n    \\"\\"\\"\\n    meta = load_meta(source, processed_dir)\\n    return meta[\\"splits\\"]\\n\\n\\n# === AutoIntent format conversion ===\\n\\ndef to_autointent_format(data: dict) -> list[dict]:\\n    \\"\\"\\"\\n    Конвертирует standard формат в AutoIntent формат.\\n\\n    Args:\\n        data: {\\"texts\\": list[str], \\"labels\\": list[int]} с OOS=-1\\n\\n    Returns:\\n        list[dict] — in-scope: {\\"utterance\\": str, \\"label\\": int}\\n                     OOS: {\\"utterance\\": str} (без поля label)\\n    \\"\\"\\"\\n    result = []\\n    for text, label in zip(data[\\"texts\\"], data[\\"labels\\"]):\\n        if label == OOS_LABEL:\\n            result.append({\\"utterance\\": text})\\n        else:\\n            result.append({\\"utterance\\": text, \\"label\\": label})\\n    return result\\n\\n\\ndef load_split_autointent(\\n    source: str,\\n    split: str,\\n    processed_dir: Path | None = None,\\n) -> list[dict]:\\n    \\"\\"\\"\\n    Загружает сплит в формате AutoIntent.\\n\\n    Args:\\n        source: \\"standard\\" | \\"deeppavlov\\"\\n        split: \\"train\\" | \\"validation\\" | \\"test\\"\\n        processed_dir: путь к data/processed/ (опционально)\\n\\n    Returns:\\n        list[dict] в формате AutoIntent\\n    \\"\\"\\"\\n    data = load_split(source, split, processed_dir)\\n    return to_autointent_format(data)\\n\\n\\ndef load_fewshot_autointent(\\n    source: str,\\n    n_shots: int,\\n    seed: int,\\n    processed_dir: Path | None = None,\\n) -> list[dict]:\\n    \\"\\"\\"\\n    Загружает few-shot выборку в формате AutoIntent.\\n\\n    Args:\\n        source: \\"standard\\" | \\"deeppavlov\\"\\n        n_shots: 10 | 20 | 50\\n        seed: 42 | 123 | 456\\n        processed_dir: путь к data/processed/ (опционально)\\n\\n    Returns:\\n        list[dict] в формате AutoIntent\\n    \\"\\"\\"\\n    data = load_fewshot(source, n_shots, seed, processed_dir)\\n    return to_autointent_format(data)\\n", "tasks/oos_detection/src/experiment_runner.py": "from __future__ import annotations\\n\\nimport contextlib\\nimport io\\nimport json\\nimport logging\\nimport os\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nfrom .data_utils import load_fewshot, load_split\\nfrom .evaluation import EvaluationResult, Evaluator\\nfrom .framework_wrappers import create_wrapper\\n\\nLOGGER = logging.getLogger(__name__)\\nOOS_TASK_DIR = Path(__file__).resolve().parent.parent\\nDEFAULT_RESULTS_FILE = OOS_TASK_DIR / \\"results\\" / \\"metrics.json\\"\\n\\n\\ndef build_wrapper(framework_name: str, **kwargs: Any):\\n    \\"\\"\\"Build framework wrapper by name.\\"\\"\\"\\n    return create_wrapper(framework_name, **kwargs)\\n\\n\\ndef load_experiment_data(\\n    source: str,\\n    mode: str,\\n    n_shots: int | None = None,\\n    seed: int | None = None,\\n) -> tuple[dict, dict, dict]:\\n    \\"\\"\\"\\n    Load train/validation/test data for one experiment.\\n\\n    Validation and test are always taken from full split.\\n    \\"\\"\\"\\n    if mode == \\"fewshot\\":\\n        if n_shots is None or seed is None:\\n            raise ValueError(\\"fewshot mode requires n_shots and seed\\")\\n        train_data = load_fewshot(source, n_shots=n_shots, seed=seed)\\n    elif mode == \\"full\\":\\n        train_data = load_split(source, split=\\"train\\")\\n    else:\\n        raise ValueError(\\"mode must be either \'full\' or \'fewshot\'\\")\\n\\n    val_data = load_split(source, split=\\"validation\\")\\n    test_data = load_split(source, split=\\"test\\")\\n    return train_data, val_data, test_data\\n\\n\\ndef _resolve_results_file(results_file: str | Path | None) -> Path:\\n    path = Path(results_file) if results_file is not None else DEFAULT_RESULTS_FILE\\n    if not path.is_absolute():\\n        path = Path.cwd() / path\\n    path.parent.mkdir(parents=True, exist_ok=True)\\n    return path\\n\\n\\ndef run_single_experiment(\\n    framework_name: str,\\n    source: str,\\n    mode: str,\\n    n_shots: int | None = None,\\n    seed: int | None = None,\\n    results_file: str | Path | None = None,\\n    wrapper_kwargs: dict[str, Any] | None = None,\\n    calibrate_threshold: bool = True,\\n    prediction_mode: str = \\"threshold\\",\\n    n_thresholds: int = 50,\\n) -> EvaluationResult:\\n    \\"\\"\\"Run one experiment and save result through existing Evaluator.\\"\\"\\"\\n    train_data, val_data, test_data = load_experiment_data(\\n        source=source,\\n        mode=mode,\\n        n_shots=n_shots,\\n        seed=seed,\\n    )\\n\\n    kwargs = {\\"prediction_mode\\": prediction_mode, **(wrapper_kwargs or {})}\\n    wrapper = build_wrapper(framework_name, **kwargs)\\n    quiet_fit = os.environ.get(\\"OOS_QUIET_FIT\\", \\"\\").lower() in (\\"1\\", \\"true\\", \\"yes\\")\\n    if quiet_fit:\\n        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):\\n            wrapper.fit(train_data[\\"texts\\"], train_data[\\"labels\\"])\\n    else:\\n        wrapper.fit(train_data[\\"texts\\"], train_data[\\"labels\\"])\\n\\n    if prediction_mode == \\"argmax\\":\\n        calibrate_threshold = False\\n\\n    if calibrate_threshold:\\n        threshold = wrapper.calibrate_threshold(\\n            val_data[\\"texts\\"],\\n            val_data[\\"labels\\"],\\n            n_thresholds=n_thresholds,\\n        )\\n        LOGGER.info(\\n            \\"Calibrated threshold for %s (%s/%s): %.4f\\",\\n            framework_name,\\n            source,\\n            mode if mode == \\"full\\" else f\\"{n_shots}shot_seed{seed}\\",\\n            threshold,\\n        )\\n\\n    mode_str = \\"full\\" if mode == \\"full\\" else f\\"{n_shots}shot\\"\\n    result_path = _resolve_results_file(results_file)\\n    evaluator = Evaluator(test_data=test_data, results_dir=result_path.parent)\\n    evaluator.metrics_file = result_path\\n    if quiet_fit:\\n        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):\\n            result = evaluator.evaluate(\\n                model=wrapper,\\n                model_name=wrapper.model_name,\\n                mode=mode_str,\\n                n_shots=n_shots,\\n                seed=seed,\\n            )\\n    else:\\n        result = evaluator.evaluate(\\n            model=wrapper,\\n            model_name=wrapper.model_name,\\n            mode=mode_str,\\n            n_shots=n_shots,\\n            seed=seed,\\n        )\\n    result.extra = {\\n        \\"framework\\": framework_name,\\n        \\"source\\": source,\\n        \\"prediction_mode\\": prediction_mode,\\n        **result.extra,\\n    }\\n    if is_degenerate_result(result):\\n        if framework_name == \\"h2o\\" and hasattr(wrapper, \\"release\\"):\\n            wrapper.release()\\n        raise RuntimeError(\\n            f\\"Degenerate predictions for {framework_name} ({mode_str}, seed={seed}): \\"\\n            f\\"in_domain_acc={result.in_domain_acc:.4f}, oos_recall={result.oos_recall:.4f}\\"\\n        )\\n    evaluator.save(result)\\n    _log_evaluation_result(result)\\n    if framework_name == \\"h2o\\" and hasattr(wrapper, \\"release\\"):\\n        wrapper.release()\\n    return result\\n\\n\\ndef is_degenerate_result(result: EvaluationResult | dict) -> bool:\\n    \\"\\"\\"Detect collapsed predict-all-OOS style metrics.\\"\\"\\"\\n    if isinstance(result, EvaluationResult):\\n        rec = result.to_dict()\\n    else:\\n        rec = result\\n    ind = rec.get(\\"in_domain_acc\\")\\n    oos_r = rec.get(\\"oos_recall\\")\\n    auroc = rec.get(\\"auroc\\")\\n    if ind is not None and oos_r is not None and ind < 0.5 and oos_r > 0.9:\\n        return True\\n    if auroc is not None and auroc <= 0.55 and oos_r is not None and oos_r > 0.9:\\n        return True\\n    return False\\n\\n\\ndef _log_evaluation_result(result: EvaluationResult) -> None:\\n    \\"\\"\\"Print metrics to stdout (compact on Kaggle via OOS_METRICS_LOG=compact).\\"\\"\\"\\n    payload = result.to_dict()\\n    line = json.dumps(payload, ensure_ascii=False, default=str)\\n    LOGGER.info(\\"METRICS_RECORD %s\\", line)\\n    if os.environ.get(\\"OOS_METRICS_LOG\\", \\"\\").lower() == \\"compact\\":\\n        print(\\"RUN_FINISH \\" + line)\\n        return\\n    print(f\\"\\\\n{\'=\' * 72}\\\\nMETRICS_RECORD\\\\n{\'=\' * 72}\\")\\n    print(json.dumps(payload, ensure_ascii=False, indent=2, default=str))\\n    print(\\"=\\" * 72)\\n\\n\\ndef run_framework_grid(\\n    frameworks: list[str],\\n    sources: list[str],\\n    run_full: bool,\\n    run_fewshot: bool,\\n    n_shots_list: list[int],\\n    seeds: list[int],\\n    results_file: str | Path | None = None,\\n    continue_on_error: bool = False,\\n    wrapper_kwargs: dict[str, Any] | None = None,\\n    calibrate_threshold: bool = True,\\n    prediction_mode: str = \\"threshold\\",\\n) -> tuple[list[EvaluationResult], list[dict[str, Any]]]:\\n    \\"\\"\\"\\n    Run framework/source/mode grid. On failure logs metadata and continues.\\n    \\"\\"\\"\\n    if not run_full and not run_fewshot:\\n        raise ValueError(\\"At least one mode must be enabled: run_full or run_fewshot\\")\\n\\n    results: list[EvaluationResult] = []\\n    errors: list[dict[str, Any]] = []\\n\\n    jobs: list[dict[str, Any]] = []\\n    for framework in frameworks:\\n        for source in sources:\\n            if run_full:\\n                jobs.append(\\n                    {\\n                        \\"framework_name\\": framework,\\n                        \\"source\\": source,\\n                        \\"mode\\": \\"full\\",\\n                        \\"n_shots\\": None,\\n                        \\"seed\\": None,\\n                    }\\n                )\\n            if run_fewshot:\\n                for n_shots in n_shots_list:\\n                    for seed in seeds:\\n                        jobs.append(\\n                            {\\n                                \\"framework_name\\": framework,\\n                                \\"source\\": source,\\n                                \\"mode\\": \\"fewshot\\",\\n                                \\"n_shots\\": n_shots,\\n                                \\"seed\\": seed,\\n                            }\\n                        )\\n\\n    for job in jobs:\\n        try:\\n            LOGGER.info(\\n                \\"Running: framework=%s source=%s mode=%s n_shots=%s seed=%s\\",\\n                job[\\"framework_name\\"],\\n                job[\\"source\\"],\\n                job[\\"mode\\"],\\n                job[\\"n_shots\\"],\\n                job[\\"seed\\"],\\n            )\\n            result = run_single_experiment(\\n                results_file=results_file,\\n                wrapper_kwargs=wrapper_kwargs,\\n                calibrate_threshold=calibrate_threshold,\\n                prediction_mode=prediction_mode,\\n                **job,\\n            )\\n            results.append(result)\\n        except Exception as exc:\\n            error_payload = {\\n                \\"framework\\": job[\\"framework_name\\"],\\n                \\"source\\": job[\\"source\\"],\\n                \\"mode\\": job[\\"mode\\"],\\n                \\"n_shots\\": job[\\"n_shots\\"],\\n                \\"seed\\": job[\\"seed\\"],\\n                \\"error\\": str(exc),\\n            }\\n            errors.append(error_payload)\\n            LOGGER.exception(\\"Experiment failed: %s\\", error_payload)\\n            if not continue_on_error:\\n                raise\\n\\n    return results, errors\\n", "tasks/oos_detection/src/framework_wrappers/__init__.py": "from .autogluon_wrapper import AutoGluonWrapper\\nfrom .base import BaseFrameworkWrapper\\nfrom .h2o_wrapper import H2OWrapper\\nfrom .lama_wrapper import LAMAWrapper\\nfrom .registry import WRAPPER_REGISTRY, create_wrapper\\n\\n__all__ = [\\n    \\"BaseFrameworkWrapper\\",\\n    \\"AutoGluonWrapper\\",\\n    \\"H2OWrapper\\",\\n    \\"LAMAWrapper\\",\\n    \\"WRAPPER_REGISTRY\\",\\n    \\"create_wrapper\\",\\n]\\n", "tasks/oos_detection/src/framework_wrappers/base.py": "from __future__ import annotations\\n\\nfrom abc import ABC, abstractmethod\\nfrom typing import Literal\\n\\nimport numpy as np\\n\\nfrom tasks.oos_detection.src.metrics import f1_oos\\n\\nPredictionMode = Literal[\\"argmax\\", \\"threshold\\"]\\n\\n\\nclass BaseFrameworkWrapper(ABC):\\n    \\"\\"\\"Common interface for framework-based OOS wrappers.\\"\\"\\"\\n\\n    oos_label: int = -1\\n\\n    def __init__(\\n        self,\\n        model_name: str,\\n        default_threshold: float = 0.5,\\n        prediction_mode: PredictionMode = \\"threshold\\",\\n    ):\\n        if prediction_mode not in (\\"argmax\\", \\"threshold\\"):\\n            raise ValueError(f\\"prediction_mode must be \'argmax\' or \'threshold\', got {prediction_mode!r}\\")\\n        self.prediction_mode: PredictionMode = prediction_mode\\n        self.model_name = model_name\\n        self.default_threshold = default_threshold\\n        self.threshold_: float | None = None\\n\\n    def _train_texts_labels(\\n        self,\\n        train_texts: list[str],\\n        train_labels: list[int],\\n    ) -> tuple[list[str], list[int]]:\\n        \\"\\"\\"Argmax: OOS as a regular class; threshold: in-scope only.\\"\\"\\"\\n        if self.prediction_mode == \\"argmax\\":\\n            return list(train_texts), list(train_labels)\\n        x_texts = [t for t, y in zip(train_texts, train_labels) if y != self.oos_label]\\n        y_labels = [y for y in train_labels if y != self.oos_label]\\n        if not x_texts:\\n            raise ValueError(\\"No in-domain samples after filtering OOS labels.\\")\\n        return x_texts, y_labels\\n\\n    @abstractmethod\\n    def fit(self, train_texts: list[str], train_labels: list[int]) -> \\"BaseFrameworkWrapper\\":\\n        \\"\\"\\"Fit wrapper on training data.\\"\\"\\"\\n\\n    @abstractmethod\\n    def predict(self, texts: list[str]) -> np.ndarray:\\n        \\"\\"\\"Predict final labels with OOS handling.\\"\\"\\"\\n\\n    @abstractmethod\\n    def predict_proba(self, texts: list[str]) -> np.ndarray:\\n        \\"\\"\\"Return OOS score for each text (higher = more OOS).\\"\\"\\"\\n\\n    @abstractmethod\\n    def _predict_in_domain(self, texts: list[str]) -> np.ndarray:\\n        \\"\\"\\"Predict in-domain labels only.\\"\\"\\"\\n\\n    def _effective_threshold(self) -> float:\\n        return self.threshold_ if self.threshold_ is not None else self.default_threshold\\n\\n    def calibrate_threshold(\\n        self,\\n        val_texts: list[str],\\n        val_labels: list[int],\\n        n_thresholds: int = 50,\\n    ) -> float:\\n        \\"\\"\\"\\n        Calibrate OOS threshold on validation split by maximizing F1 OOS.\\n        \\"\\"\\"\\n        if not val_texts:\\n            raise ValueError(\\"Validation texts are empty, cannot calibrate threshold.\\")\\n\\n        y_true = np.asarray(val_labels)\\n        oos_scores = np.asarray(self.predict_proba(val_texts), dtype=float)\\n        y_in_domain = np.asarray(self._predict_in_domain(val_texts))\\n\\n        if oos_scores.size == 0:\\n            raise ValueError(\\"predict_proba returned no OOS scores for validation texts.\\")\\n\\n        score_std = float(np.std(oos_scores))\\n        if score_std < 1e-6:\\n            raise RuntimeError(\\n                \\"OOS scores degenerate (std < 1e-6) — model likely failed to train or scores are constant.\\"\\n            )\\n\\n        thresholds = np.linspace(float(oos_scores.min()), float(oos_scores.max()), n_thresholds)\\n        best_threshold = float(thresholds[0])\\n        best_f1 = -1.0\\n\\n        for threshold in thresholds:\\n            y_pred = np.where(oos_scores >= threshold, self.oos_label, y_in_domain)\\n            score = f1_oos(y_true, y_pred, oos_label=self.oos_label)\\n            if score > best_f1:\\n                best_f1 = score\\n                best_threshold = float(threshold)\\n\\n        self.threshold_ = best_threshold\\n        return best_threshold\\n", "tasks/oos_detection/src/framework_wrappers/registry.py": "from __future__ import annotations\\n\\nfrom typing import Any\\n\\nfrom .autogluon_wrapper import AutoGluonWrapper\\nfrom .base import BaseFrameworkWrapper\\nfrom .h2o_wrapper import H2OWrapper\\nfrom .lama_wrapper import LAMAWrapper\\n\\nWRAPPER_REGISTRY = {\\n    \\"autogluon\\": AutoGluonWrapper,\\n    \\"h2o\\": H2OWrapper,\\n    \\"lama\\": LAMAWrapper,\\n}\\n\\n\\ndef create_wrapper(name: str, **kwargs: Any) -> BaseFrameworkWrapper:\\n    \\"\\"\\"Create framework wrapper by registry name.\\"\\"\\"\\n    key = name.lower()\\n    if key not in WRAPPER_REGISTRY:\\n        valid = \\", \\".join(sorted(WRAPPER_REGISTRY))\\n        raise ValueError(f\\"Unknown framework \'{name}\'. Expected one of: {valid}\\")\\n    return WRAPPER_REGISTRY[key](**kwargs)\\n", "tasks/oos_detection/src/framework_wrappers/autogluon_wrapper.py": "from __future__ import annotations\\n\\nimport logging\\n\\nimport numpy as np\\nimport pandas as pd\\nfrom sentence_transformers import SentenceTransformer\\n\\nfrom .base import BaseFrameworkWrapper\\n\\nLOGGER = logging.getLogger(__name__)\\n\\n\\nclass AutoGluonWrapper(BaseFrameworkWrapper):\\n    \\"\\"\\"\\n    AutoGluon TabularPredictor wrapper for OOS detection.\\n\\n    Uses pre-trained sentence embeddings (multilingual-e5-large-instruct) as features\\n    and trains AutoGluon TabularPredictor on the resulting tabular vectors.\\n    OOS score = 1 - max_class_probability.\\n    \\"\\"\\"\\n\\n    def __init__(\\n        self,\\n        default_threshold: float = 0.5,\\n        embedder_name: str = \\"intfloat/multilingual-e5-large-instruct\\",\\n        time_limit: int | None = 600,\\n        num_cpus: int = 1,\\n        seed: int = 42,\\n        prediction_mode: str = \\"threshold\\",\\n    ):\\n        suffix = \\"_argmax\\" if prediction_mode == \\"argmax\\" else \\"_threshold\\"\\n        super().__init__(\\n            model_name=f\\"autogluon{suffix}\\",\\n            default_threshold=default_threshold,\\n            prediction_mode=prediction_mode,\\n        )\\n        self.embedder_name = embedder_name\\n        self.time_limit = time_limit\\n        self.num_cpus = num_cpus\\n        self.seed = seed\\n\\n        self._embedder: SentenceTransformer | None = None\\n        self._predictor = None\\n        self._classes: np.ndarray | None = None\\n        self._feature_names: list[str] = []\\n\\n    def _get_embedder(self) -> SentenceTransformer:\\n        if self._embedder is None:\\n            self._embedder = SentenceTransformer(self.embedder_name)\\n        return self._embedder\\n\\n    def _embed(self, texts: list[str]) -> np.ndarray:\\n        model = self._get_embedder()\\n        return np.asarray(\\n            model.encode(\\n                texts,\\n                normalize_embeddings=True,\\n                show_progress_bar=False,\\n            )\\n        )\\n\\n    def fit(self, train_texts: list[str], train_labels: list[int]) -> \\"AutoGluonWrapper\\":\\n        try:\\n            from autogluon.tabular import TabularPredictor\\n        except ImportError as exc:\\n            raise ImportError(\\n                \\"AutoGluon is not installed. Install autogluon.tabular to run this wrapper.\\"\\n            ) from exc\\n\\n        x_texts, y_labels = self._train_texts_labels(train_texts, train_labels)\\n\\n        embeddings = self._embed(x_texts)\\n        self._feature_names = [f\\"f_{idx}\\" for idx in range(embeddings.shape[1])]\\n        self._classes = np.array(sorted(set(y_labels)))\\n\\n        train_df = pd.DataFrame(embeddings, columns=self._feature_names)\\n        train_df[\\"label\\"] = y_labels\\n\\n        self._predictor = TabularPredictor(\\n            label=\\"label\\",\\n            problem_type=\\"multiclass\\",\\n            learner_kwargs={\\"random_state\\": self.seed},\\n        )\\n        self._predictor.fit(\\n            train_data=train_df,\\n            time_limit=self.time_limit,\\n            ag_args_fit={\\"num_cpus\\": self.num_cpus},\\n            verbosity=0,\\n        )\\n        lb = self._predictor.leaderboard(extra_info=False, silent=True)\\n        if lb is None or len(lb) == 0:\\n            raise RuntimeError(\\n                \\"AutoGluon produced an empty leaderboard — training likely failed.\\"\\n            )\\n\\n        LOGGER.info(\\n            \\"AutoGluon fit done on %d in-domain samples using %s embeddings.\\",\\n            len(x_texts),\\n            self.embedder_name,\\n        )\\n        return self\\n\\n    def _predict_in_domain(self, texts: list[str]) -> np.ndarray:\\n        if self._predictor is None:\\n            raise RuntimeError(\\"Model is not fitted.\\")\\n        embeddings = self._embed(texts)\\n        test_df = pd.DataFrame(embeddings, columns=self._feature_names)\\n        preds = self._predictor.predict(test_df)\\n        return np.asarray(preds.tolist() if hasattr(preds, \\"tolist\\") else list(preds))\\n\\n    def predict_proba(self, texts: list[str]) -> np.ndarray:\\n        if self._predictor is None:\\n            raise RuntimeError(\\"Model is not fitted.\\")\\n        embeddings = self._embed(texts)\\n        test_df = pd.DataFrame(embeddings, columns=self._feature_names)\\n        proba_df = self._predictor.predict_proba(test_df)\\n        proba = proba_df.to_numpy()\\n        if self.prediction_mode == \\"argmax\\" and self._classes is not None:\\n            classes = list(self._classes)\\n            if self.oos_label in classes:\\n                oos_idx = classes.index(self.oos_label)\\n                return proba[:, oos_idx]\\n        return 1.0 - proba.max(axis=1)\\n\\n    def predict(self, texts: list[str]) -> np.ndarray:\\n        if self.prediction_mode == \\"argmax\\":\\n            return self._predict_in_domain(texts)\\n        oos_scores = self.predict_proba(texts)\\n        in_domain_preds = self._predict_in_domain(texts)\\n        threshold = self._effective_threshold()\\n        return np.where(oos_scores >= threshold, self.oos_label, in_domain_preds)\\n", "tasks/oos_detection/src/framework_wrappers/h2o_wrapper.py": "from __future__ import annotations\\n\\nimport contextlib\\nimport io\\nimport logging\\nimport os\\nimport warnings\\n\\nimport numpy as np\\nimport pandas as pd\\nfrom sentence_transformers import SentenceTransformer\\n\\nfrom .base import BaseFrameworkWrapper\\n\\nLOGGER = logging.getLogger(__name__)\\n\\n# In-scope intents are 0..149; OOS (-1) is encoded as 150 for H2O (avoids factor \\"0\\" = OOS bug).\\nH2O_OOS_INTERNAL = 150\\n\\n\\ndef _h2o_quiet_enabled() -> bool:\\n    for key in (\\"OOS_QUIET_FIT\\", \\"OOS_H2O_QUIET\\"):\\n        if os.environ.get(key, \\"\\").lower() in (\\"1\\", \\"true\\", \\"yes\\"):\\n            return True\\n    return False\\n\\n\\n@contextlib.contextmanager\\ndef _suppress_h2o_output():\\n    \\"\\"\\"Hide H2O cluster banner, progress bars, and pandas conversion warnings.\\"\\"\\"\\n    if not _h2o_quiet_enabled():\\n        yield\\n        return\\n    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):\\n        with warnings.catch_warnings():\\n            warnings.simplefilter(\\"ignore\\")\\n            yield\\n\\n\\nclass H2OWrapper(BaseFrameworkWrapper):\\n    \\"\\"\\"\\n    H2O AutoML wrapper for OOS detection.\\n\\n    Uses pre-trained sentence embeddings as features and H2O AutoML on tabular vectors.\\n    Labels are encoded as ints 0..149 (in-scope) and 150 (OOS) — never string factors.\\n    \\"\\"\\"\\n\\n    def __init__(\\n        self,\\n        default_threshold: float = 0.5,\\n        embedder_name: str = \\"intfloat/multilingual-e5-large-instruct\\",\\n        max_models: int = 5,\\n        max_runtime_secs: int = 1500,\\n        seed: int = 42,\\n        prediction_mode: str = \\"threshold\\",\\n    ):\\n        suffix = \\"_argmax\\" if prediction_mode == \\"argmax\\" else \\"_threshold\\"\\n        super().__init__(\\n            model_name=f\\"h2o{suffix}\\",\\n            default_threshold=default_threshold,\\n            prediction_mode=prediction_mode,\\n        )\\n        self.embedder_name = embedder_name\\n        self.max_models = max_models\\n        self.max_runtime_secs = max_runtime_secs\\n        self.seed = seed\\n\\n        self._embedder: SentenceTransformer | None = None\\n        self._aml = None\\n        self._feature_names: list[str] = []\\n        self._h2o_initialized: bool = False\\n        self._oos_internal = H2O_OOS_INTERNAL\\n\\n    def __del__(self):\\n        self.release()\\n\\n    @staticmethod\\n    def _encode_label(label: int, oos_label: int = -1) -> int:\\n        return H2O_OOS_INTERNAL if label == oos_label else int(label)\\n\\n    def _decode_label(self, raw) -> int:\\n        val = int(float(raw))\\n        return self.oos_label if val == self._oos_internal else val\\n\\n    def _get_embedder(self) -> SentenceTransformer:\\n        if self._embedder is None:\\n            self._embedder = SentenceTransformer(self.embedder_name)\\n        return self._embedder\\n\\n    def _embed(self, texts: list[str]) -> np.ndarray:\\n        model = self._get_embedder()\\n        return np.asarray(\\n            model.encode(\\n                texts,\\n                normalize_embeddings=True,\\n                show_progress_bar=False,\\n            )\\n        )\\n\\n    def _init_h2o(self) -> None:\\n        import h2o\\n\\n        with _suppress_h2o_output():\\n            try:\\n                h2o.cluster().shutdown(prompt=False)\\n            except Exception:\\n                pass\\n            h2o.init(\\n                strict_version_check=False,\\n                log_level=\\"ERRR\\",\\n                nthreads=int(os.environ.get(\\"H2O_NTHREADS\\", \\"2\\")),\\n                max_mem_size=os.environ.get(\\"H2O_MAX_MEM\\", \\"6G\\"),\\n            )\\n        self._h2o_initialized = True\\n\\n    def fit(self, train_texts: list[str], train_labels: list[int]) -> \\"H2OWrapper\\":\\n        try:\\n            import h2o\\n            from h2o.automl import H2OAutoML\\n        except ImportError as exc:\\n            raise ImportError(\\"H2O is not installed. Install h2o to run this wrapper.\\") from exc\\n\\n        x_texts, y_labels = self._train_texts_labels(train_texts, train_labels)\\n        y_internal = [self._encode_label(label) for label in y_labels]\\n\\n        embeddings = self._embed(x_texts)\\n        self._feature_names = [f\\"f_{idx}\\" for idx in range(embeddings.shape[1])]\\n\\n        frame = pd.DataFrame(embeddings, columns=self._feature_names)\\n        frame[\\"label\\"] = y_internal\\n\\n        self._init_h2o()\\n        with _suppress_h2o_output():\\n            train_h2o = h2o.H2OFrame(frame)\\n            train_h2o[\\"label\\"] = train_h2o[\\"label\\"].asfactor()\\n\\n            aml = H2OAutoML(\\n                max_models=self.max_models,\\n                max_runtime_secs=self.max_runtime_secs,\\n                seed=self.seed,\\n                sort_metric=\\"logloss\\",\\n                balance_classes=False,\\n            )\\n            aml.train(x=self._feature_names, y=\\"label\\", training_frame=train_h2o)\\n        if aml.leader is None:\\n            raise RuntimeError(\\n                \\"H2O AutoML produced no leader model (empty or failed run). \\"\\n                \\"Try increasing max_runtime_secs or check the training frame.\\"\\n            )\\n        self._aml = aml\\n        LOGGER.info(\\"H2O fit done on %d samples.\\", len(x_texts))\\n        return self\\n\\n    def release(self) -> None:\\n        \\"\\"\\"Shut down H2O cluster to free JVM heap between Kaggle runs.\\"\\"\\"\\n        if not self._h2o_initialized:\\n            return\\n        try:\\n            import h2o\\n\\n            with _suppress_h2o_output():\\n                h2o.cluster().shutdown(prompt=False)\\n        except Exception:\\n            pass\\n        self._h2o_initialized = False\\n        self._aml = None\\n\\n    def _predict_frame(self, texts: list[str]) -> pd.DataFrame:\\n        if self._aml is None:\\n            raise RuntimeError(\\"Model is not fitted.\\")\\n\\n        import h2o\\n\\n        embeddings = self._embed(texts)\\n        frame = pd.DataFrame(embeddings, columns=self._feature_names)\\n        with _suppress_h2o_output():\\n            test_h2o = h2o.H2OFrame(frame)\\n            pred_h2o = self._aml.leader.predict(test_h2o)\\n            return pred_h2o.as_data_frame(use_pandas=True)\\n\\n    def _predict_in_domain(self, texts: list[str]) -> np.ndarray:\\n        pred_df = self._predict_frame(texts)\\n        return np.asarray([self._decode_label(x) for x in pred_df[\\"predict\\"]])\\n\\n    def predict_proba(self, texts: list[str]) -> np.ndarray:\\n        pred_df = self._predict_frame(texts)\\n        oos_col = f\\"p{self._oos_internal}\\"\\n        if oos_col in pred_df.columns:\\n            return pred_df[oos_col].to_numpy(dtype=float)\\n        proba_cols = [c for c in pred_df.columns if c != \\"predict\\"]\\n        if not proba_cols:\\n            raise RuntimeError(\\"H2O prediction does not contain class probabilities.\\")\\n        proba = pred_df[proba_cols].to_numpy(dtype=float)\\n        return 1.0 - proba.max(axis=1)\\n\\n    def predict(self, texts: list[str]) -> np.ndarray:\\n        if self.prediction_mode == \\"argmax\\":\\n            return self._predict_in_domain(texts)\\n        oos_scores = self.predict_proba(texts)\\n        in_domain_preds = self._predict_in_domain(texts)\\n        threshold = self._effective_threshold()\\n        return np.where(oos_scores >= threshold, self.oos_label, in_domain_preds)\\n", "tasks/oos_detection/src/framework_wrappers/lama_wrapper.py": "from __future__ import annotations\\n\\nimport logging\\nimport os\\n\\n# Mac CPU settings to avoid multiprocessing fork/spawn issues and OMP oversubscription\\nos.environ.setdefault(\\"OMP_NUM_THREADS\\", \\"1\\")\\nos.environ.setdefault(\\"MKL_NUM_THREADS\\", \\"1\\")\\nos.environ.setdefault(\\"OPENBLAS_NUM_THREADS\\", \\"1\\")\\n\\nimport numpy as np\\nimport pandas as pd\\nfrom sentence_transformers import SentenceTransformer\\n\\nfrom .base import BaseFrameworkWrapper\\n\\nLOGGER = logging.getLogger(__name__)\\n\\n\\nclass LAMAWrapper(BaseFrameworkWrapper):\\n    \\"\\"\\"\\n    LightAutoML (LAMA) wrapper for OOS detection.\\n\\n    LAMA is treated as LightAutoML tabular classification over text embeddings.\\n    We use `intfloat/multilingual-e5-large-instruct` for text representation.\\n    \\"\\"\\"\\n\\n    def __init__(\\n        self,\\n        default_threshold: float = 0.5,\\n        embedder_name: str = \\"intfloat/multilingual-e5-large-instruct\\",\\n        timeout: int = 600,\\n        cpu_limit: int = 1,\\n        seed: int = 42,\\n        prediction_mode: str = \\"threshold\\",\\n    ):\\n        suffix = \\"_argmax\\" if prediction_mode == \\"argmax\\" else \\"_threshold\\"\\n        super().__init__(\\n            model_name=f\\"lama{suffix}\\",\\n            default_threshold=default_threshold,\\n            prediction_mode=prediction_mode,\\n        )\\n        self.embedder_name = embedder_name\\n        self.timeout = timeout\\n        self.cpu_limit = cpu_limit\\n        self.seed = seed\\n\\n        self._embedder: SentenceTransformer | None = None\\n        self._automl = None\\n        self._class_labels: list[int] = []\\n        self._feature_names: list[str] = []\\n\\n    def _get_embedder(self) -> SentenceTransformer:\\n        if self._embedder is None:\\n            self._embedder = SentenceTransformer(self.embedder_name)\\n        return self._embedder\\n\\n    def _embed(self, texts: list[str]) -> np.ndarray:\\n        model = self._get_embedder()\\n        return np.asarray(\\n            model.encode(\\n                texts,\\n                normalize_embeddings=True,\\n                show_progress_bar=False,\\n            )\\n        )\\n\\n    def fit(self, train_texts: list[str], train_labels: list[int]) -> \\"LAMAWrapper\\":\\n        try:\\n            from lightautoml.automl.presets.tabular_presets import TabularAutoML\\n            from lightautoml.tasks import Task\\n        except ImportError as exc:\\n            raise ImportError(\\n                \\"LightAutoML is not installed. Install lightautoml to run lama wrapper.\\"\\n            ) from exc\\n\\n        x_texts, y_labels = self._train_texts_labels(train_texts, train_labels)\\n\\n        embeddings = self._embed(x_texts)\\n        self._feature_names = [f\\"f_{idx}\\" for idx in range(embeddings.shape[1])]\\n        self._class_labels = sorted(set(y_labels))\\n        label_to_index = {label: idx for idx, label in enumerate(self._class_labels)}\\n        y_internal = [label_to_index[label] for label in y_labels]\\n\\n        train_df = pd.DataFrame(embeddings, columns=self._feature_names)\\n        train_df[\\"label\\"] = y_internal\\n\\n        task = Task(\\"multiclass\\")\\n        automl = TabularAutoML(\\n            task=task,\\n            timeout=self.timeout,\\n            cpu_limit=self.cpu_limit,\\n            reader_params={\\"random_state\\": self.seed},\\n        )\\n        automl.fit_predict(train_df, roles={\\"target\\": \\"label\\"}, verbose=0)\\n        self._automl = automl\\n        LOGGER.info(\\"LAMA fit done on %d in-domain samples using %s.\\", len(x_texts), self.embedder_name)\\n        return self\\n\\n    def _predict_proba_matrix(self, texts: list[str]) -> np.ndarray:\\n        if self._automl is None:\\n            raise RuntimeError(\\"Model is not fitted.\\")\\n        embeddings = self._embed(texts)\\n        test_df = pd.DataFrame(embeddings, columns=self._feature_names)\\n        pred = self._automl.predict(test_df)\\n        proba = pred.data if hasattr(pred, \\"data\\") else pred\\n        return np.asarray(proba)\\n\\n    def _predict_in_domain(self, texts: list[str]) -> np.ndarray:\\n        proba = self._predict_proba_matrix(texts)\\n        idx = proba.argmax(axis=1)\\n        return np.asarray([self._class_labels[i] for i in idx])\\n\\n    def predict_proba(self, texts: list[str]) -> np.ndarray:\\n        proba = self._predict_proba_matrix(texts)\\n        if self.prediction_mode == \\"argmax\\" and self.oos_label in self._class_labels:\\n            oos_idx = self._class_labels.index(self.oos_label)\\n            return proba[:, oos_idx]\\n        return 1.0 - proba.max(axis=1)\\n\\n    def predict(self, texts: list[str]) -> np.ndarray:\\n        if self.prediction_mode == \\"argmax\\":\\n            return self._predict_in_domain(texts)\\n        oos_scores = self.predict_proba(texts)\\n        in_domain_preds = self._predict_in_domain(texts)\\n        threshold = self._effective_threshold()\\n        return np.where(oos_scores >= threshold, self.oos_label, in_domain_preds)\\n", "tasks/oos_detection/scripts/__init__.py": "", "tasks/oos_detection/scripts/prepare_data.py": "#!/usr/bin/env python3\\n\\"\\"\\"\\nПодготовка данных CLINC150 для экспериментов.\\n\\nИсточники:\\n    - standard: github.com/clinc/oos-eval (100 OOS train, для воспроизводимости ADB/DETER)\\n    - deeppavlov: HuggingFace DeepPavlov/clinc150 (200 OOS train, для AutoIntent)\\n\\nИспользование:\\n    python prepare_data.py --source standard\\n    python prepare_data.py --source deeppavlov\\n    python prepare_data.py --source all\\n\\nВыходные файлы:\\n    data/processed/{source}/full.json     — полные сплиты\\n    data/processed/{source}/fewshot.json  — few-shot выборки\\n    data/processed/{source}/meta.json     — метаданные\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport json\\nimport urllib.request\\nfrom pathlib import Path\\n\\nimport numpy as np\\n\\n\\n# === Constants ===\\n\\nOOS_LABEL = -1\\nN_INTENTS = 150\\nN_SHOTS_LIST = [10, 20, 50]\\nSEEDS = [42, 123, 456]\\nOOS_RATIO = 0.1\\n\\nSTANDARD_URL = \\"https://raw.githubusercontent.com/clinc/oos-eval/master/data/data_full.json\\"\\n\\nSCRIPT_DIR = Path(__file__).parent\\nDATA_DIR = SCRIPT_DIR.parent / \\"data\\"\\nRAW_DIR = DATA_DIR / \\"raw\\"\\nPROCESSED_DIR = DATA_DIR / \\"processed\\"\\n\\n\\n# === Download functions ===\\n\\ndef download_standard() -> dict:\\n    \\"\\"\\"Download data from github.com/clinc/oos-eval.\\"\\"\\"\\n    cache_path = RAW_DIR / \\"standard_data_full.json\\"\\n\\n    if cache_path.exists():\\n        print(f\\"Using cached: {cache_path}\\")\\n        with open(cache_path, \\"r\\", encoding=\\"utf-8\\") as f:\\n            return json.load(f)\\n\\n    print(f\\"Downloading from {STANDARD_URL}...\\")\\n    RAW_DIR.mkdir(parents=True, exist_ok=True)\\n\\n    with urllib.request.urlopen(STANDARD_URL) as response:\\n        data = json.loads(response.read().decode(\\"utf-8\\"))\\n\\n    with open(cache_path, \\"w\\", encoding=\\"utf-8\\") as f:\\n        json.dump(data, f, ensure_ascii=False)\\n\\n    print(f\\"Saved to {cache_path}\\")\\n    return data\\n\\n\\ndef download_deeppavlov() -> dict:\\n    \\"\\"\\"Download data from HuggingFace DeepPavlov/clinc150.\\"\\"\\"\\n    from datasets import load_dataset\\n\\n    print(\\"Loading DeepPavlov/clinc150 from HuggingFace...\\")\\n    import os\\n    _tok = os.environ.get(\\"HF_TOKEN\\") or os.environ.get(\\"HUGGINGFACE_HUB_TOKEN\\")\\n    _kw = {\\"token\\": _tok} if _tok else {}\\n    ds = load_dataset(\\"DeepPavlov/clinc150\\", **_kw)\\n\\n    # Use numeric IDs as intent names (0-149)\\n    intent_names = {i: f\\"intent_{i}\\" for i in range(N_INTENTS)}\\n\\n    return {\\"dataset\\": ds, \\"intent_names\\": intent_names}\\n\\n\\n# === Processing functions ===\\n\\ndef process_standard(raw_data: dict) -> tuple[dict, dict, list[dict]]:\\n    \\"\\"\\"\\n    Process standard format data.\\n\\n    Input format: {\\"train\\": [[text, label], ...], \\"oos_train\\": [[text, \\"oos\\"], ...], ...}\\n    Output format: {\\"texts\\": [...], \\"labels\\": [...]} with OOS=-1\\n    \\"\\"\\"\\n    # Collect all intent names and create mapping\\n    all_intents = set()\\n    for split in [\\"train\\", \\"val\\", \\"test\\"]:\\n        for text, label in raw_data[split]:\\n            all_intents.add(label)\\n\\n    intent_to_id = {name: idx for idx, name in enumerate(sorted(all_intents))}\\n    intents_list = [{\\"id\\": idx, \\"name\\": name} for name, idx in sorted(intent_to_id.items(), key=lambda x: x[1])]\\n\\n    # Process splits\\n    splits = {}\\n    split_mapping = {\\"train\\": \\"train\\", \\"val\\": \\"validation\\", \\"test\\": \\"test\\"}\\n\\n    for src_split, dst_split in split_mapping.items():\\n        texts = []\\n        labels = []\\n        seen = set()\\n\\n        # In-scope examples\\n        for text, label in raw_data[src_split]:\\n            if text not in seen:\\n                seen.add(text)\\n                texts.append(text)\\n                labels.append(intent_to_id[label])\\n\\n        # OOS examples\\n        oos_key = f\\"oos_{src_split}\\"\\n        if oos_key in raw_data:\\n            for text, _ in raw_data[oos_key]:\\n                if text not in seen:\\n                    seen.add(text)\\n                    texts.append(text)\\n                    labels.append(OOS_LABEL)\\n\\n        splits[dst_split] = {\\"texts\\": texts, \\"labels\\": labels}\\n\\n    # Stats\\n    stats = {}\\n    for split_name, data in splits.items():\\n        n_oos = sum(1 for l in data[\\"labels\\"] if l == OOS_LABEL)\\n        stats[split_name] = {\\n            \\"total\\": len(data[\\"texts\\"]),\\n            \\"n_inscope\\": len(data[\\"texts\\"]) - n_oos,\\n            \\"n_oos\\": n_oos\\n        }\\n\\n    return splits, stats, intents_list\\n\\n\\ndef process_deeppavlov(raw_data: dict) -> tuple[dict, dict, list[dict]]:\\n    \\"\\"\\"\\n    Process DeepPavlov/clinc150 data.\\n\\n    OOS examples have label=None in this dataset.\\n    \\"\\"\\"\\n    ds = raw_data[\\"dataset\\"]\\n    intent_names = raw_data[\\"intent_names\\"]\\n\\n    intents_list = [{\\"id\\": i, \\"name\\": intent_names[i]} for i in sorted(intent_names.keys())]\\n\\n    splits = {}\\n    stats = {}\\n\\n    for split_name in [\\"train\\", \\"validation\\", \\"test\\"]:\\n        texts = []\\n        labels = []\\n        seen = set()\\n\\n        for item in ds[split_name]:\\n            text = item[\\"utterance\\"]\\n            if text not in seen:\\n                seen.add(text)\\n                texts.append(text)\\n                if item[\\"label\\"] is None:\\n                    labels.append(OOS_LABEL)\\n                else:\\n                    labels.append(item[\\"label\\"])\\n\\n        splits[split_name] = {\\"texts\\": texts, \\"labels\\": labels}\\n\\n        n_oos = sum(1 for l in labels if l == OOS_LABEL)\\n        stats[split_name] = {\\n            \\"total\\": len(texts),\\n            \\"n_inscope\\": len(texts) - n_oos,\\n            \\"n_oos\\": n_oos\\n        }\\n\\n    return splits, stats, intents_list\\n\\n\\ndef create_fewshot_splits(train_data: dict) -> dict:\\n    \\"\\"\\"Create few-shot training splits.\\"\\"\\"\\n    texts = train_data[\\"texts\\"]\\n    labels = train_data[\\"labels\\"]\\n\\n    # Group by label\\n    label_to_indices = {}\\n    for idx, label in enumerate(labels):\\n        if label not in label_to_indices:\\n            label_to_indices[label] = []\\n        label_to_indices[label].append(idx)\\n\\n    fewshot = {}\\n\\n    for n_shots in N_SHOTS_LIST:\\n        n_key = f\\"n{n_shots}\\"\\n        fewshot[n_key] = {}\\n\\n        for seed in SEEDS:\\n            seed_key = f\\"seed{seed}\\"\\n            np.random.seed(seed)\\n\\n            sampled_texts = []\\n            sampled_labels = []\\n\\n            # Sample n_shots per in-scope intent\\n            for label in sorted(label_to_indices.keys()):\\n                if label == OOS_LABEL:\\n                    continue\\n                indices = label_to_indices[label]\\n                n_sample = min(n_shots, len(indices))\\n                sampled_idx = np.random.choice(indices, size=n_sample, replace=False)\\n                for idx in sampled_idx:\\n                    sampled_texts.append(texts[idx])\\n                    sampled_labels.append(labels[idx])\\n\\n            # Sample OOS proportionally\\n            n_oos = round(N_INTENTS * n_shots * OOS_RATIO)\\n            if OOS_LABEL in label_to_indices:\\n                oos_indices = label_to_indices[OOS_LABEL]\\n                n_oos_sample = min(n_oos, len(oos_indices))\\n                sampled_oos_idx = np.random.choice(oos_indices, size=n_oos_sample, replace=False)\\n                for idx in sampled_oos_idx:\\n                    sampled_texts.append(texts[idx])\\n                    sampled_labels.append(labels[idx])\\n\\n            fewshot[n_key][seed_key] = {\\n                \\"texts\\": sampled_texts,\\n                \\"labels\\": sampled_labels\\n            }\\n\\n    return fewshot\\n\\n\\ndef save_processed(source: str, splits: dict, fewshot: dict, stats: dict, intents: list[dict]):\\n    \\"\\"\\"Save processed data to disk.\\"\\"\\"\\n    output_dir = PROCESSED_DIR / source\\n    output_dir.mkdir(parents=True, exist_ok=True)\\n\\n    # Full splits\\n    full_path = output_dir / \\"full.json\\"\\n    with open(full_path, \\"w\\", encoding=\\"utf-8\\") as f:\\n        json.dump(splits, f, ensure_ascii=False, indent=2)\\n\\n    # Few-shot splits\\n    fewshot_path = output_dir / \\"fewshot.json\\"\\n    with open(fewshot_path, \\"w\\", encoding=\\"utf-8\\") as f:\\n        json.dump(fewshot, f, ensure_ascii=False, indent=2)\\n\\n    # Metadata\\n    meta = {\\n        \\"source\\": source,\\n        \\"n_intents\\": N_INTENTS,\\n        \\"oos_label\\": OOS_LABEL,\\n        \\"splits\\": stats,\\n        \\"fewshot\\": {\\n            \\"n_shots\\": N_SHOTS_LIST,\\n            \\"seeds\\": SEEDS,\\n            \\"oos_ratio\\": OOS_RATIO\\n        },\\n        \\"intents\\": intents\\n    }\\n    meta_path = output_dir / \\"meta.json\\"\\n    with open(meta_path, \\"w\\", encoding=\\"utf-8\\") as f:\\n        json.dump(meta, f, ensure_ascii=False, indent=2)\\n\\n    print(f\\"Saved to {output_dir}/\\")\\n\\n\\ndef prepare_source(source: str):\\n    \\"\\"\\"Prepare data for a single source.\\"\\"\\"\\n    print(f\\"\\\\n{\'=\'*50}\\")\\n    print(f\\"Preparing: {source}\\")\\n    print(\\"=\\"*50)\\n\\n    # Download\\n    if source == \\"standard\\":\\n        raw_data = download_standard()\\n        splits, stats, intents = process_standard(raw_data)\\n    else:  # deeppavlov\\n        raw_data = download_deeppavlov()\\n        splits, stats, intents = process_deeppavlov(raw_data)\\n\\n    # Print stats\\n    for split_name, s in stats.items():\\n        print(f\\"{split_name:12}: {s[\'n_inscope\']:5} in-scope + {s[\'n_oos\']:4} OOS = {s[\'total\']:5} total\\")\\n\\n    # Create few-shot splits\\n    fewshot = create_fewshot_splits(splits[\\"train\\"])\\n    print(f\\"\\\\nFew-shot splits: n={N_SHOTS_LIST}, seeds={SEEDS}\\")\\n\\n    # Save\\n    save_processed(source, splits, fewshot, stats, intents)\\n\\n\\ndef main():\\n    parser = argparse.ArgumentParser(description=\\"Prepare CLINC150 data\\")\\n    parser.add_argument(\\n        \\"--source\\",\\n        choices=[\\"standard\\", \\"deeppavlov\\", \\"all\\"],\\n        default=\\"all\\",\\n        help=\\"Data source: standard (github/clinc), deeppavlov (HuggingFace), or all\\"\\n    )\\n    args = parser.parse_args()\\n\\n    sources = [\\"standard\\", \\"deeppavlov\\"] if args.source == \\"all\\" else [args.source]\\n\\n    for source in sources:\\n        prepare_source(source)\\n\\n    print(\\"\\\\nDone!\\")\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n"}')
for rel, content in BUNDLE_FILES.items():
    p = BUNDLE_ROOT / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content, encoding='utf-8')
sys.path.insert(0, str(BUNDLE_ROOT))



In [ ]:
# 4. Prepare data (prepare_data.py — deeppavlov)
import contextlib
import io

from tasks.oos_detection.scripts.prepare_data import prepare_source

_proc = BUNDLE_ROOT / "tasks" / "oos_detection" / "data" / "processed" / SOURCE
if not (_proc / "full.json").exists():
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        prepare_source(SOURCE)


In [ ]:
# 5. Pending argmax grid (12 jobs — built from local metrics.json)
import json
import logging
from pathlib import Path

from tasks.oos_detection.src.experiment_runner import is_degenerate_result, run_single_experiment

logging.disable(logging.CRITICAL)

MODEL_BY_FW = {"h2o": "h2o_argmax", "lama": "lama_argmax", "autogluon": "autogluon_argmax"}
PENDING_JOBS = json.loads('[{"framework": "h2o", "mode": "fewshot", "n_shots": 10, "seed": 42, "mode_str": "10shot"}, {"framework": "h2o", "mode": "fewshot", "n_shots": 10, "seed": 123, "mode_str": "10shot"}, {"framework": "h2o", "mode": "fewshot", "n_shots": 10, "seed": 456, "mode_str": "10shot"}, {"framework": "lama", "mode": "full", "n_shots": null, "seed": 123, "mode_str": "full"}, {"framework": "lama", "mode": "fewshot", "n_shots": 10, "seed": 123, "mode_str": "10shot"}, {"framework": "lama", "mode": "fewshot", "n_shots": 10, "seed": 456, "mode_str": "10shot"}, {"framework": "lama", "mode": "fewshot", "n_shots": 20, "seed": 42, "mode_str": "20shot"}, {"framework": "lama", "mode": "fewshot", "n_shots": 20, "seed": 123, "mode_str": "20shot"}, {"framework": "lama", "mode": "fewshot", "n_shots": 20, "seed": 456, "mode_str": "20shot"}, {"framework": "lama", "mode": "fewshot", "n_shots": 50, "seed": 42, "mode_str": "50shot"}, {"framework": "lama", "mode": "fewshot", "n_shots": 50, "seed": 123, "mode_str": "50shot"}, {"framework": "lama", "mode": "fewshot", "n_shots": 50, "seed": 456, "mode_str": "50shot"}]')

def _wrapper_kwargs(framework, mode, n_shots):
    kw = {"embedder_name": EMBEDDER, "prediction_mode": "argmax"}
    if framework == "h2o":
        kw["max_models"] = 5
        kw["max_runtime_secs"] = 900 if mode == "full" else 600
    elif framework == "lama":
        kw["cpu_limit"] = 1
        if mode == "full":
            kw["timeout"] = 2100
        elif n_shots == 10:
            kw["timeout"] = 900
        elif n_shots == 20:
            kw["timeout"] = 1200
        elif n_shots == 50:
            kw["timeout"] = 1500
        else:
            kw["timeout"] = 900
    elif framework == "autogluon":
        kw["time_limit"] = 900 if mode == "full" else 600
    return kw

def _purge_bad_h2o():
    if not METRICS_JSON.exists():
        return
    rows = json.loads(METRICS_JSON.read_text(encoding="utf-8"))
    kept = [r for r in rows if not (r.get("model_name") == "h2o_argmax" and is_degenerate_result(r))]
    if len(kept) != len(rows):
        METRICS_JSON.write_text(json.dumps(kept, ensure_ascii=False, indent=2), encoding="utf-8")

def _already_done(model_name, mode_str, seed):
    if not METRICS_JSON.exists():
        return False
    for r in json.loads(METRICS_JSON.read_text(encoding="utf-8")):
        if (r.get("model_name"), r.get("mode"), r.get("seed")) != (model_name, mode_str, seed):
            continue
        if (r.get("extra") or {}).get("prediction_mode") != "argmax":
            continue
        if is_degenerate_result(r) or r.get("f1_oos") is None:
            continue
        return True
    return False

_purge_bad_h2o()

if not PENDING_JOBS:
    print("RUN_FINISH " + json.dumps({"status": "nothing_pending", "jobs": 0}, ensure_ascii=False))
else:
    total = len(PENDING_JOBS)
    for i, job in enumerate(PENDING_JOBS, 1):
        fw = job["framework"]
        mode = job["mode"]
        n_shots = job["n_shots"]
        seed = job["seed"]
        mode_str = job["mode_str"]
        model_name = MODEL_BY_FW[fw]
        meta = {
            "run": i, "total": total, "framework": fw, "mode": mode_str,
            "n_shots": n_shots, "seed": seed, "prediction_mode": "argmax",
            "embedder": EMBEDDER, "source": SOURCE,
        }
        print("RUN_START " + json.dumps(meta, ensure_ascii=False))
        if _already_done(model_name, mode_str, seed):
            print("RUN_FINISH " + json.dumps({**meta, "status": "skipped"}, ensure_ascii=False))
            continue
        try:
            run_single_experiment(
                framework_name=fw,
                source=SOURCE,
                mode=mode,
                n_shots=n_shots,
                seed=seed,
                results_file=METRICS_JSON,
                wrapper_kwargs={**_wrapper_kwargs(fw, mode, n_shots), "seed": seed},
                calibrate_threshold=False,
                prediction_mode="argmax",
            )
        except Exception as e:
            print("RUN_FINISH " + json.dumps({
                **meta, "status": "failed",
                "error_type": type(e).__name__, "error_message": str(e),
            }, ensure_ascii=False))
            if fw == "h2o":
                try:
                    import h2o
                    h2o.cluster().shutdown(prompt=False)
                except Exception:
                    pass
